In [2]:
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
import numpy as np
import pandas as pd
import statsmodels.api as sm
import scipy.stats as stats
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, roc_auc_score
from scipy.stats import chi2

# Suprimir warnings específicos
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

#### Base Enade

“Quais características sociodemográficas, familiares e educacionais estão associadas à faixa de renda familiar dos estudantes do ENADE?”

In [3]:
def convert_num(x):
    try:
        return float(x.replace(',', '.').strip())
    except:
        return pd.NA

df_base = pd.read_csv(
        'C:\\Users\\Matheus\\Desktop\\Arquivos_PUC\\Analise Descritiva e Probabilidade\\Projeto final\\Projeto final\\MICRODADOS_ENADE_2017.txt', 
        sep=';', 
        encoding='latin1',
        converters={
            'NT_OBJ_CE': convert_num,
            'NT_GER': convert_num
        }
    )

C:\Users\Matheus\AppData\Local\Temp\ipykernel_47752\2791999572.py:7: DtypeWarning: Columns (29,31,32,45,46,47,54,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df_base = pd.read_csv(


In [4]:
df_base

,NU_ANO,CO_IES,CO_CATEGAD,CO_ORGACAD,CO_GRUPO,CO_CURSO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO,CO_REGIAO_CURSO,...,QE_I72,QE_I73,QE_I74,QE_I75,QE_I76,QE_I77,QE_I78,QE_I79,QE_I80,QE_I81
0,2017,1,1,10028,5710,3,1,5103403,51,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017,1,1,10028,5710,3,1,5103403,51,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017,1,1,10028,5710,3,1,5103403,51,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017,1,1,10028,5710,3,1,5103403,51,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017,1,1,10028,5710,3,1,5103403,51,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537431,2017,19578,2,10022,6208,5001279,1,3513009,35,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
537432,2017,19578,2,10022,6208,5001279,1,3513009,35,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
537433,2017,19578,2,10022,6208,5001279,1,3513009,35,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
537434,2017,19578,2,10022,6208,5001279,1,3513009,35,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Entendimento da base

In [5]:
df_base.columns

Index(['NU_ANO', 'CO_IES', 'CO_CATEGAD', 'CO_ORGACAD', 'CO_GRUPO', 'CO_CURSO',
       'CO_MODALIDADE', 'CO_MUNIC_CURSO', 'CO_UF_CURSO', 'CO_REGIAO_CURSO',
       ...
       'QE_I72', 'QE_I73', 'QE_I74', 'QE_I75', 'QE_I76', 'QE_I77', 'QE_I78',
       'QE_I79', 'QE_I80', 'QE_I81'],
      dtype='object', length=150)

In [6]:
df_base.shape

(537436, 150)

In [7]:
df_base.describe()

,NU_ANO,CO_IES,CO_CATEGAD,CO_ORGACAD,CO_GRUPO,CO_CURSO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO,CO_REGIAO_CURSO,...,QE_I59,QE_I60,QE_I61,QE_I62,QE_I63,QE_I64,QE_I65,QE_I66,QE_I67,QE_I68
count,537436.0,537436.000000,537436.000000,537436.000000,537436.000000,5.374360e+05,537436.000000,5.374360e+05,537436.000000,537436.000000,...,467656.000000,467656.000000,467656.000000,467656.000000,467656.000000,467656.000000,467656.000000,467656.000000,467656.000000,467656.000000
mean,2017.0,1385.499157,3.345377,10025.231575,3192.639719,3.748719e+05,0.786739,3.401669e+06,33.836092,3.042857,...,4.999589,4.954471,4.912284,4.942768,5.006193,5.137623,5.157195,5.284339,4.965849,4.976962
std,0.0,2504.935866,1.567353,3.484124,2044.616513,6.854908e+05,0.409611,8.874856e+05,8.867267,0.981309,...,1.377580,1.544319,1.427220,1.569085,1.544688,1.296523,1.589632,1.214273,1.590165,1.534717
min,2017.0,1.000000,1.000000,10019.000000,21.000000,3.000000e+00,0.000000,1.100023e+06,11.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2017.0,322.000000,2.000000,10022.000000,2001.000000,3.447600e+04,1.000000,3.106200e+06,31.000000,3.000000,...,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000,4.000000,5.000000,4.000000,4.000000
50%,2017.0,583.000000,4.000000,10028.000000,2402.000000,9.750700e+04,1.000000,3.509502e+06,35.000000,3.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,6.000000,6.000000,6.000000,5.000000,6.000000
75%,2017.0,1472.000000,5.000000,10028.000000,5710.000000,3.113030e+05,1.000000,4.106902e+06,41.000000,4.000000,...,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
max,2017.0,19739.000000,7.000000,10028.000000,6409.000000,5.001385e+06,1.000000,5.300108e+06,53.000000,5.000000,...,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000


#### Variáveis

In [58]:
df_modelo = df_base[
    [
        'QE_I08', # Qual a renda total de sua família, incluindo seus rendimentos?
        'NU_IDADE', # Idade do inscrito em 26/11/2017
        'TP_SEXO', # Tipo de sexo
        'QE_I02', # Qual é a sua cor ou raça?
        'QE_I04', # Até que etapa de escolarização seu pai concluiu?
        'QE_I05', # Até que etapa de escolarização sua mãe concluiu?
        'QE_I21', # Alguém em sua família concluiu um curso superior?
        'QE_I17', # Em que tipo de escola você cursou o ensino médio?
        'QE_I18', # Qual modalidade de ensino médio você concluiu?
        'QE_I16', # Em que unidade da Federação você concluiu o ensino médio?
        'QE_I10' # Qual alternativa a seguir melhor descreve sua situação de trabalho (exceto estágio ou bolsas)?
    ]
]

-   estudantes com maior escolaridade parental têm maior probabilidade de pertencer às faixas mais altas de renda

-   estudantes oriundos de escola privada têm maior chance de estar nas faixas mais altas de renda

-   estudantes que trabalham têm distribuição distinta entre faixas de renda

-   há diferenças regionais na composição da renda familiar dos estudantes

In [59]:
df_modelo.isna().sum()

QE_I08      69780
NU_IDADE        0
TP_SEXO         0
QE_I02      69780
QE_I04      69780
QE_I05      69780
QE_I21      69780
QE_I17      69780
QE_I18      69780
QE_I16      69780
QE_I10      69780
dtype: int64

In [60]:
df_modelo.dropna(inplace=True)

In [61]:
cor_raca = {
    'A': 'Branca',
    'B': 'Preta',
    'C': 'Amarela',
    'D': 'Parda',
    'E': 'Indígena',
    'F': 'Não quis declarar'
}

df_modelo['QE_I02'] = df_modelo['QE_I02'].map(cor_raca)

In [62]:
escolaridade = {
    'A': 'Nenhuma',
    'B': 'Ensino Fundamental',
    'C': 'Ensino Fundamental',
    'D': 'Ensino Médio',
    'E': 'Ensino Superior',
    'F': 'Pós-Graduação'
}

In [63]:
df_modelo['QE_I04'] = df_modelo['QE_I04'].map(escolaridade)
df_modelo['QE_I05'] = df_modelo['QE_I05'].map(escolaridade)

In [64]:
renda = {
    'A': 'Baixa',
    'B': 'Baixa',
    'C': 'Média',
    'D': 'Média',
    'E': 'Alta',
    'F': 'Alta'
}

In [65]:
df_modelo['QE_I08'] = df_modelo['QE_I08'].map(renda)

In [66]:
curso_superior = {
    'A': 'Sim',
    'B': 'Não'    
}

In [67]:
df_modelo['QE_I21'] = df_modelo['QE_I21'].map(curso_superior)

In [68]:
tipo_em = {
    "A": "Publica",
    "B": "Privada",
    "C": "Internacional",
    "D": "Publica",
    "E": "Privada",
    "F": "Internacional"
}

In [69]:
df_modelo['QE_I17'] = df_modelo['QE_I17'].map(tipo_em)

In [70]:
modalidade_em = {
    "A": "Ensino médio tradicional",
    "B": "Profissionalizante técnico (eletrônica, contabilidade, agrícola, outro)",
    "C": "Profissionalizante magistério (Curso Normal)",
    "D": "Educação de Jovens e Adultos (EJA) e/ou Supletivo",
    "E": "Outra modalidade"
}

In [71]:
df_modelo['QE_I18'] = df_modelo['QE_I18'].map(modalidade_em)

In [72]:
estados_ibge = {
    11: "Rondônia (RO)",
    12: "Acre (AC)",
    13: "Amazonas (AM)",
    14: "Roraima (RR)",
    15: "Pará (PA)",
    16: "Amapá (AP)",
    17: "Tocantins (TO)",
    21: "Maranhão (MA)",
    22: "Piauí (PI)",
    23: "Ceará (CE)",
    24: "Rio Grande do Norte (RN)",
    25: "Paraíba (PB)",
    26: "Pernambuco (PE)",
    27: "Alagoas (AL)",
    28: "Sergipe (SE)",
    29: "Bahia (BA)",
    31: "Minas Gerais (MG)",
    32: "Espírito Santo (ES)",
    33: "Rio de Janeiro (RJ)",
    35: "São Paulo (SP)",
    41: "Paraná (PR)",
    42: "Santa Catarina (SC)",
    43: "Rio Grande do Sul (RS)",
    50: "Mato Grosso do Sul (MS)",
    51: "Mato Grosso (MT)",
    52: "Goiás (GO)",
    53: "Distrito Federal (DF)",
    99: "Não se aplica"
}

In [73]:
df_modelo['QE_I16'] = df_modelo['QE_I16'].map(estados_ibge)

In [74]:
situacao_trabalho = {
    "A": "Desocupado",
    "B": "Ocupação Parcial",
    "C": "Ocupação Parcial",
    "D": "Ocupação Parcial",
    "E": "Ocupação Integral"
}

In [76]:
df_modelo['QE_I10'] = df_modelo['QE_I10'].map(situacao_trabalho)

In [77]:
df_modelo

,QE_I08,NU_IDADE,TP_SEXO,QE_I02,QE_I04,QE_I05,QE_I21,QE_I17,QE_I18,QE_I16,QE_I10
0,Baixa,26,F,Branca,Ensino Médio,Pós-Graduação,Sim,Privada,Ensino médio tradicional,Mato Grosso (MT),Ocupação Parcial
1,Baixa,23,F,Parda,Ensino Superior,Ensino Médio,Sim,Publica,"Profissionalizante técnico (eletrônica, contab...",Mato Grosso (MT),Ocupação Integral
2,Alta,23,M,Parda,Ensino Superior,Pós-Graduação,Sim,Privada,Ensino médio tradicional,Mato Grosso (MT),Desocupado
3,Baixa,23,M,Branca,Ensino Médio,Ensino Médio,Sim,Publica,Ensino médio tradicional,Rondônia (RO),Desocupado
4,Alta,24,M,Branca,Ensino Médio,Pós-Graduação,Sim,Privada,Ensino médio tradicional,Mato Grosso (MT),Ocupação Parcial
...,...,...,...,...,...,...,...,...,...,...,...
537430,Média,36,M,Parda,Ensino Fundamental,Ensino Fundamental,Sim,Publica,Ensino médio tradicional,São Paulo (SP),Ocupação Integral
537431,Média,26,M,Parda,Ensino Fundamental,Ensino Fundamental,Não,Publica,Ensino médio tradicional,São Paulo (SP),Ocupação Integral
537432,Baixa,41,M,Branca,Ensino Fundamental,Ensino Médio,Sim,Publica,Ensino médio tradicional,São Paulo (SP),Desocupado
537433,Baixa,22,M,Parda,Ensino Fundamental,Ensino Médio,Não,Publica,Ensino médio tradicional,São Paulo (SP),Ocupação Integral
